# Lab 2: Produce a grounded HR answer

## Business problem

The proposed RAG system is intended to reduce the time employees wait for routine HR answers while protecting trust in the information. Retrieval returns policy text, but the assistant must turn it into a direct answer with evidence, identify its sources, and refuse to invent missing policy.

## Mission

Connect the Lab 1 index to an LLM while keeping retrieval, context, generation, and citations visible.

## What this lab covers

Use the searchable policy index to produce a grounded HR answer. You will retrieve evidence, build the context, generate a cited response, handle insufficient evidence, and record what happened during the request.

Run each cell in order. The comments explain what the code is doing and why the step matters.

## Exercise 1: Use the approved handbook index

**Mission:** Use the same approved source, chunking settings, metadata, and embedding model as Lab 1.

**Why it matters:** Changing any of these creates a different index and makes retrieval results difficult to compare.

### Load the handbook source

**Mission:** Load the approved handbook source before retrieving evidence.

Run the next cell, inspect its output, and only then continue.

In [ ]:
# Define the chunking rules before applying them.
# The lab uses characters so the result is easy to inspect.
# Production systems often use token limits, but the reasoning is the same.
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
EMBEDDING_MODEL = "text-embedding-3-small"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

### Load the shared company handbook PDF

**Mission:** Load the same company handbook used in Lab 1 before creating the answer workflow.

The PDF is the approved company policy source for this project. Lab 2 uses its extracted pages, so retrieval and answers use the same information as Lab 1.

Run the next cell, inspect its output, and only then continue.

In [ ]:
# Load the PDF into one record per page.
# The loader extracts text and keeps page metadata for citations later.
pdf_path = Path("sample_documents/company_handbook.pdf")
pages = PyPDFLoader(str(pdf_path)).load()

sections = []
for page in pages:
    sections.append({
        "source": pdf_path.name,
        "section": f"Page {page.metadata.get('page', 0) + 1}",
        "text": page.page_content,
    })

print("Handbook pages loaded:", len(sections))

### Configure the chunker

**Mission:** Choose the chunking method, chunk size and overlap before creating chunks.

Common chunking methods you may see in RAG projects:

- **Fixed character size:** cut every set number of characters. It is simple and predictable, but it can cut through a sentence or policy rule.
- **Token-based:** cut by the model's token count. This matches the model limit more closely, but token boundaries are less visible when learning.
- **Sentence-based:** keep complete sentences together until the size limit is reached. This protects meaning but may create uneven chunk sizes.
- **Heading or section-based:** keep a complete policy section together when possible. This is useful for well-structured handbooks.
- **Recursive splitting:** try paragraphs, lines, sentences, words, and finally characters in that order. This is the practical default used in this lab because it balances readable boundaries with a size limit.
- **Semantic chunking:** compare neighboring sentences and split when the topic changes. It can help mixed-topic documents, but it needs extra processing and evaluation.

The overlap carries a small amount of text from one chunk into the next so a rule is less likely to be separated at the boundary.

Run the next cell, inspect its output, and only then continue.

In [ ]:
# Define the chunking rules before applying them.
# The lab uses characters so the result is easy to inspect.
# Production systems often use token limits, but the reasoning is the same.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    # Try paragraph, line, sentence, word, then character boundaries in that order.
    # This helps keep each chunk readable before making a smaller cut.
    separators=["\n\n", "\n", ". ", " ", ""],
)

documents = []
for section in sections:
    pieces = splitter.split_text(section["text"])
    for position, piece in enumerate(pieces):
        metadata = {
            "source": section["source"], "section": section["section"],
            "position": position, "version": "2026.1", "status": "current",
        }
        documents.append(Document(page_content=piece, metadata=metadata))

### Build the searchable index

**Mission:** Embed the prepared chunks and store them in the vector store.

Run the next cell, inspect its output, and only then continue.

In [ ]:
# Wrap each chunk and its metadata as a searchable document.
# The local store keeps the mechanics visible while matching the Lab 1 index contract.
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="hr_handbook_lab2",
)
print("Indexed chunks:", len(documents))

## Exercise 2: Retrieve before generating

**Mission:** Make the evidence visible before asking an LLM to answer.

In [ ]:
# Search for the chunks most similar to the employee question.
# TOP_K controls how many candidates we inspect before generation.
QUESTION = "Can I expense a $300 train ticket without approval?"
TOP_K = 3

retrieved = vector_store.similarity_search_with_score(QUESTION, k=TOP_K)

for document, score in retrieved:
    print("Similarity:", round(score, 3))
    print(document.metadata["source"], "|", document.metadata["section"])
    print(document.page_content)
    print()

### Retrieval checkpoint

Do not generate until the travel approval section is present. Generation cannot recover evidence that retrieval failed to provide.

## Exercise 3: Build the grounded context

**Mission:** Combine the retrieved text with human-readable citations.

In [ ]:
# Build the context that will be supplied to the generation model.
# Keep the source and section beside each passage so the answer can cite it.
context_parts = []

for document, score in retrieved:
    citation = document.metadata["source"] + " | " + document.metadata["section"]
    context_parts.append("Source: " + citation + "\n" + document.page_content)

context = ""
for part in context_parts:
    context = context + part + "\n\n"
print(context)

## Exercise 4: Generate the answer

**Mission:** Ask the generation model to answer only from the retrieved policy.

OpenAI creates embeddings and Claude generates the answer. They have separate responsibilities. Replacing either one requires evaluation.

**Industry choices:** This lab uses Anthropic Claude for generation so students see a multi-provider RAG pattern. Common choices in current systems include OpenAI models, Anthropic Claude, Google Gemini, and hosted open-weight models through AWS Bedrock, Azure AI Foundry, or Google Vertex AI. The important LLMOps practice is to record the model name and evaluate a replacement before release.

In [ ]:
# The model receives the question plus retrieved evidence.
# The instruction asks it to stay grounded in that evidence.
from anthropic import Anthropic

client = Anthropic()
GENERATION_MODEL = "claude-haiku-4-5"

system_message = (
    "You are the company HR policy assistant. "
    "Answer only from the supplied context. "
    "If the answer is absent, say: I cannot find that answer in the available HR policy. "
    "End supported answers with the source and section used."
)

user_message = "Context:\n" + context + "\n\nEmployee question:\n" + QUESTION
response = client.messages.create(
    model=GENERATION_MODEL, max_tokens=300, system=system_message,
    messages=[{"role": "user", "content": user_message}],
)
print(response.content[0].text)

## Exercise 5: Test missing evidence

**Mission:** Confirm that the application does not invent a policy.

In [ ]:
# Search for the chunks most similar to the employee question.
# TOP_K controls how many candidates we inspect before generation.
MISSING_QUESTION = "Does the company reimburse home internet service?"
missing_docs = vector_store.similarity_search(MISSING_QUESTION, k=TOP_K)

missing_parts = []
for document in missing_docs:
    citation = document.metadata["source"] + " | " + document.metadata["section"]
    missing_parts.append("Source: " + citation + "\n" + document.page_content)

missing_context = ""
for part in missing_parts:
    missing_context = missing_context + part + "\n\n"
missing_message = "Context:\n" + missing_context + "\n\nEmployee question:\n" + MISSING_QUESTION
response = client.messages.create(
    model=GENERATION_MODEL, max_tokens=300, system=system_message,
    messages=[{"role": "user", "content": missing_message}],
)
print(response.content[0].text)

## Exercise 6: Record the request path

**Mission:** Capture enough evidence to troubleshoot one request.

Record the index version, embedding model, generation model, retrieved sources, and elapsed time in production. Avoid logging sensitive employee text unless the data policy permits it.

In [ ]:
# Record the configuration and timing for this request.
# Operational records help explain what happened when an answer is questioned.
request_record = {
    "index_version": "2026.1",
    "embedding_model": EMBEDDING_MODEL,
    "generation_model": GENERATION_MODEL,
    "retrieved_sources": [document.metadata["source"] for document, score in retrieved],
}

print(request_record)

## Exercise 7: Make the request observable

**Mission:** Record the information an operator needs to investigate one answer.

Logs should show the path through the system without storing sensitive employee text by default.

In [ ]:
# Record the configuration and timing for this request.
# Operational records help explain what happened when an answer is questioned.
request_record = {
    "request_id": "demo-001",
    "index_version": "2026.1",
    "embedding_model": EMBEDDING_MODEL,
    "generation_model": GENERATION_MODEL,
    "retrieved_sources": [document.metadata["source"] for document, score in retrieved],
    "retrieved_sections": [document.metadata["section"] for document, score in retrieved],
}

print(request_record)

## Exercise 8: Test an unsupported question

**Mission:** Confirm that the assistant does not pretend that nearby text is an answer.

A production assistant needs an explicit insufficient-evidence response, not only a successful example.

In [ ]:
# Search for the chunks most similar to the employee question.
# TOP_K controls how many candidates we inspect before generation.
UNSUPPORTED_QUESTION = "Does the company pay for home internet service?"
unsupported = vector_store.similarity_search(UNSUPPORTED_QUESTION, k=TOP_K)

print("Evidence found:", len(unsupported))
for document in unsupported:
    print(document.metadata["source"], "|", document.metadata["section"])

print("The answer layer must still verify that this evidence answers the question.")

## Exercise 9: Clarify a vague question

**Mission:** See why a specific question gives the retriever better search terms.

This exercise uses a human-written rewrite. We are not asking another model to rewrite the question yet. In a later agentic system, query rewriting could become an automated step.

In [ ]:
vague_question = "What about salaries?"
clear_question = "How does an employee request a compensation review?"

print("Original question:", vague_question)
print("Clear search question:", clear_question)

# Now use the clearer question with the same retriever.
clear_results = vector_store.similarity_search(clear_question, k=TOP_K)

for document in clear_results:
    print(document.metadata["source"], "|", document.metadata["section"])
    print(document.page_content)


## Exercise 10: Add context to a short chunk

**Mission:** See how the ingestion pipeline can add document context automatically before embedding.

No employee or operator should read every chunk and add context by hand. The ingestion code or managed knowledge-base service performs this step for every document. Context can come from the file name, title, section, page, version, and other metadata.

In [ ]:
short_chunk = "Section 2 says the limit is 15 days."
document_title = "HR Policy 2026"
section_name = "Time Off"
section_number = "Section 2: Annual leave"

# This small function represents an automated ingestion step.
contextual_chunk = document_title + " | " + section_name + " | " + section_number + " | " + short_chunk

print("Standalone:", short_chunk)
print("Contextual:", contextual_chunk)

# The contextual version is the text that could be embedded and indexed.

### Lab 2 checkpoint

The request path is now visible: question, retrieve, inspect, optionally rewrite, optionally add context, build context, generate, cite, and record the request. A fluent answer is not proof that retrieval succeeded.

## Production handoff: managed retrieval and generation

The local notebook makes retrieval and prompt construction visible. In production, the source may be SharePoint, Confluence, or S3, and a managed knowledge-base service may handle synchronization, parsing, chunking, embeddings, and vector indexing.

The application still owns the user identity, access scope, prompt rules, citations, insufficient-evidence response, logging, and cost controls.